# EDA Bronze Amazon 2023 trên Google Colab

Notebook này đọc trực tiếp tầng Bronze từ Hugging Face dataset `chuongdo1104/amazon-2023-bronze`, gồm:

- `bronze/bronze_train.parquet/`
- `bronze/bronze_val.parquet/`
- `bronze/bronze_test.parquet/`
- `bronze/bronze_meta.parquet`

Mục tiêu: EDA từng folder/split, kiểm tra chất lượng dữ liệu, phân bố tương tác user/item, k-core, cold-start, metadata, và đặc biệt phân bố Head/Mid/Tail để chọn ngưỡng popularity.

In [ ]:
# Cell 1 - Cài thư viện cho Colab
!pip -q install huggingface_hub duckdb pyarrow pandas numpy matplotlib seaborn tqdm

In [ ]:
# Cell 2 - Imports và cấu hình hiển thị
from pathlib import Path
import json
import math
import os
import re

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download, list_repo_files
from IPython.display import display, Markdown
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titleweight'] = 'bold'

REPO_ID = 'chuongdo1104/amazon-2023-bronze'
REPO_TYPE = 'dataset'
HF_ROOT = 'bronze'
LOCAL_DIR = Path('/content/amazon_2023_bronze')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train': 'bronze/bronze_train.parquet',
    'val': 'bronze/bronze_val.parquet',
    'test': 'bronze/bronze_test.parquet',
}
META_PATH = 'bronze/bronze_meta.parquet'

display(Markdown(f'**Dataset:** `{REPO_ID}`  \n**Local cache:** `{LOCAL_DIR}`'))

In [ ]:
# Cell 3 - Liệt kê và tải các file parquet từ Hugging Face
repo_files = list_repo_files(REPO_ID, repo_type=REPO_TYPE)
bronze_files = sorted([f for f in repo_files if f.startswith('bronze/')])

display(pd.DataFrame({'path': bronze_files}).head(50))
print(f'Tổng số file/folder entry trong bronze: {len(bronze_files):,}')

def parquet_files_under(prefix: str):
    prefix = prefix.rstrip('/') + '/'
    return sorted([f for f in bronze_files if f.startswith(prefix) and f.endswith('.parquet')])

downloaded = {}
for split, prefix in SPLITS.items():
    files = parquet_files_under(prefix)
    if not files:
        raise FileNotFoundError(f'Không thấy parquet part trong {prefix}')
    local_paths = []
    for f in tqdm(files, desc=f'Download {split}'):
        local_paths.append(hf_hub_download(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            filename=f,
            local_dir=LOCAL_DIR,
            local_dir_use_symlinks=False,
        ))
    downloaded[split] = local_paths

meta_local = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename=META_PATH,
    local_dir=LOCAL_DIR,
    local_dir_use_symlinks=False,
)

download_summary = pd.DataFrame([
    {'artifact': split, 'parquet_files': len(paths), 'local_glob': str(LOCAL_DIR / SPLITS[split] / '*.parquet')}
    for split, paths in downloaded.items()
] + [{'artifact': 'meta', 'parquet_files': 1, 'local_glob': meta_local}])
display(download_summary)

In [ ]:
# Cell 4 - Tạo DuckDB views cho train/val/test/meta/all
con = duckdb.connect(database=':memory:')

paths = {split: str(LOCAL_DIR / prefix / '*.parquet') for split, prefix in SPLITS.items()}
paths['meta'] = str(Path(meta_local))

for split in ['train', 'val', 'test']:
    con.execute(f"CREATE OR REPLACE VIEW {split} AS SELECT * FROM read_parquet('{paths[split]}')")

con.execute(f"CREATE OR REPLACE VIEW meta AS SELECT * FROM read_parquet('{paths['meta']}')")
con.execute("""
CREATE OR REPLACE VIEW interactions_all AS
SELECT 'train' AS split, * FROM train
UNION ALL SELECT 'val' AS split, * FROM val
UNION ALL SELECT 'test' AS split, * FROM test
""")
con.execute("CREATE OR REPLACE VIEW user_freq_all AS SELECT reviewer_id, COUNT(*) AS n_interactions FROM interactions_all GROUP BY reviewer_id")
con.execute("CREATE OR REPLACE VIEW item_freq_all AS SELECT parent_asin, COUNT(*) AS n_interactions FROM interactions_all GROUP BY parent_asin")
con.execute("CREATE OR REPLACE VIEW item_freq_train AS SELECT parent_asin, COUNT(*) AS train_freq FROM train GROUP BY parent_asin")
con.execute("CREATE OR REPLACE VIEW user_freq_train AS SELECT reviewer_id, COUNT(*) AS train_freq FROM train GROUP BY reviewer_id")

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

display(q("""
SELECT 'train' artifact, COUNT(*) rows FROM train
UNION ALL SELECT 'val', COUNT(*) FROM val
UNION ALL SELECT 'test', COUNT(*) FROM test
UNION ALL SELECT 'meta', COUNT(*) FROM meta
"""))

In [ ]:
# Cell 5 - Schema và sample từng folder/artifact
for name in ['train', 'val', 'test', 'meta']:
    display(Markdown(f'### Schema: `{name}`'))
    display(q(f"DESCRIBE {name}"))
    display(Markdown(f'### Sample rows: `{name}`'))
    display(q(f"SELECT * FROM {name} LIMIT 5"))

In [ ]:
# Cell 6 - Tổng quan split: số dòng, user, item, rating, thời gian
split_overview = q("""
SELECT
  split,
  COUNT(*) AS interactions,
  COUNT(DISTINCT reviewer_id) AS users,
  COUNT(DISTINCT parent_asin) AS items,
  ROUND(COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT reviewer_id), 0), 3) AS avg_interactions_per_user,
  ROUND(COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT parent_asin), 0), 3) AS avg_interactions_per_item,
  MIN(rating) AS min_rating,
  MAX(rating) AS max_rating,
  AVG(rating) AS avg_rating,
  MIN(to_timestamp(timestamp)) AS min_time,
  MAX(to_timestamp(timestamp)) AS max_time
FROM interactions_all
GROUP BY split
ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
""")
display(split_overview)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=split_overview, x='split', y='interactions', ax=axes[0])
axes[0].set_title('Số tương tác theo split')
sns.barplot(data=split_overview, x='split', y='users', ax=axes[1])
axes[1].set_title('Số user theo split')
sns.barplot(data=split_overview, x='split', y='items', ax=axes[2])
axes[2].set_title('Số item theo split')
for ax in axes:
    ax.ticklabel_format(axis='y', style='plain')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 - Null, ID rỗng, rating/timestamp bất thường, duplicate key
quality_sql = """
SELECT
  split,
  COUNT(*) AS rows,
  SUM(CASE WHEN reviewer_id IS NULL OR trim(reviewer_id) = '' THEN 1 ELSE 0 END) AS bad_reviewer_id,
  SUM(CASE WHEN parent_asin IS NULL OR trim(parent_asin) = '' THEN 1 ELSE 0 END) AS bad_parent_asin,
  SUM(CASE WHEN timestamp IS NULL OR timestamp <= 0 THEN 1 ELSE 0 END) AS bad_timestamp,
  SUM(CASE WHEN rating IS NULL OR rating < 1 OR rating > 5 THEN 1 ELSE 0 END) AS bad_rating,
  SUM(CASE WHEN review_title IS NULL OR trim(review_title) = '' THEN 1 ELSE 0 END) AS empty_review_title,
  SUM(CASE WHEN review_text IS NULL OR trim(review_text) = '' THEN 1 ELSE 0 END) AS empty_review_text,
  COUNT(*) - COUNT(DISTINCT reviewer_id || '|' || parent_asin) AS duplicate_user_item_rows,
  COUNT(*) - COUNT(DISTINCT reviewer_id || '|' || parent_asin || '|' || CAST(timestamp AS VARCHAR) || '|' || CAST(rating AS VARCHAR)) AS duplicate_exact_like_rows
FROM interactions_all
GROUP BY split
ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
"""
quality = q(quality_sql)
display(quality)

quality_rate = quality.copy()
for col in quality_rate.columns:
    if col not in ['split', 'rows']:
        quality_rate[col + '_pct'] = 100 * quality_rate[col] / quality_rate['rows']
display(quality_rate[['split'] + [c for c in quality_rate.columns if c.endswith('_pct')]])

In [ ]:
# Cell 8 - Missing value theo từng cột và từng artifact
def missing_report(table: str) -> pd.DataFrame:
    cols = q(f"DESCRIBE {table}")['column_name'].tolist()
    parts = []
    for c in cols:
        parts.append(f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}")
    df = q(f"SELECT COUNT(*) AS rows, {', '.join(parts)} FROM {table}")
    rows = int(df.loc[0, 'rows'])
    out = df.drop(columns=['rows']).T.reset_index()
    out.columns = ['column', 'null_count']
    out['null_pct'] = 100 * out['null_count'] / max(rows, 1)
    out.insert(0, 'artifact', table)
    return out.sort_values('null_pct', ascending=False)

missing_all = pd.concat([missing_report(t) for t in ['train', 'val', 'test', 'meta']], ignore_index=True)
display(missing_all)

plot_df = missing_all[missing_all['null_pct'] > 0].copy()
if len(plot_df):
    plt.figure(figsize=(14, max(4, 0.28 * len(plot_df))))
    sns.barplot(data=plot_df, y='column', x='null_pct', hue='artifact')
    plt.title('Tỷ lệ null theo cột')
    plt.xlabel('Null (%)')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.show()
else:
    print('Không có null trong các cột đã kiểm tra.')

In [ ]:
# Cell 9 - Phân bố rating và helpful_vote
rating_dist = q("""
SELECT split, rating, COUNT(*) AS n
FROM interactions_all
GROUP BY split, rating
ORDER BY split, rating
""")
display(rating_dist)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=rating_dist, x='rating', y='n', hue='split', ax=axes[0])
axes[0].set_title('Phân bố rating theo split')
axes[0].ticklabel_format(axis='y', style='plain')

helpful = q("""
SELECT split, helpful_vote
FROM interactions_all
WHERE helpful_vote IS NOT NULL
""")
sns.histplot(data=helpful, x='helpful_vote', hue='split', bins=80, log_scale=(False, True), ax=axes[1])
axes[1].set_title('Helpful vote histogram, log y')
plt.tight_layout()
plt.show()

display(q("""
SELECT split,
       MIN(helpful_vote) min_helpful,
       quantile_cont(helpful_vote, 0.5) p50,
       quantile_cont(helpful_vote, 0.9) p90,
       quantile_cont(helpful_vote, 0.99) p99,
       MAX(helpful_vote) max_helpful
FROM interactions_all
GROUP BY split
ORDER BY split
"""))

In [ ]:
# Cell 10 - Phân bố thời gian tương tác theo tháng
monthly = q("""
SELECT split, strftime(to_timestamp(timestamp), '%Y-%m') AS year_month, COUNT(*) AS n
FROM interactions_all
GROUP BY split, year_month
ORDER BY year_month, split
""")
display(monthly.head(20))
display(monthly.tail(20))

plt.figure(figsize=(16, 5))
sns.lineplot(data=monthly, x='year_month', y='n', hue='split', marker='o')
plt.title('Số tương tác theo tháng')
plt.xlabel('Year-month')
plt.ylabel('Interactions')
plt.xticks(rotation=75)
plt.ticklabel_format(axis='y', style='plain')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 - Phân bố số tương tác/user: % user có 1, 2, 3... tương tác
user_pct = q("""
WITH f AS (
  SELECT split, reviewer_id, COUNT(*) AS n_interactions
  FROM interactions_all
  GROUP BY split, reviewer_id
), total AS (
  SELECT split, COUNT(*) AS n_users FROM f GROUP BY split
)
SELECT f.split, f.n_interactions, COUNT(*) AS users,
       ROUND(100.0 * COUNT(*) / MAX(total.n_users), 4) AS user_pct
FROM f JOIN total USING(split)
GROUP BY f.split, f.n_interactions
ORDER BY f.split, f.n_interactions
""")
display(user_pct[user_pct['n_interactions'] <= 30])

plt.figure(figsize=(14, 5))
plot_df = user_pct[user_pct['n_interactions'] <= 30]
sns.lineplot(data=plot_df, x='n_interactions', y='user_pct', hue='split', marker='o')
plt.title('% user theo số tương tác trong từng split, cắt tại 30')
plt.xlabel('Số tương tác/user')
plt.ylabel('% user')
plt.tight_layout()
plt.show()

display(q("""
SELECT
  'all_splits' AS scope,
  MIN(n_interactions) AS min_interactions,
  quantile_cont(n_interactions, 0.25) AS p25,
  quantile_cont(n_interactions, 0.50) AS p50,
  quantile_cont(n_interactions, 0.75) AS p75,
  quantile_cont(n_interactions, 0.90) AS p90,
  quantile_cont(n_interactions, 0.95) AS p95,
  quantile_cont(n_interactions, 0.99) AS p99,
  MAX(n_interactions) AS max_interactions,
  AVG(n_interactions) AS avg_interactions
FROM user_freq_all
UNION ALL
SELECT
  'train_only', MIN(train_freq), quantile_cont(train_freq, 0.25), quantile_cont(train_freq, 0.50),
  quantile_cont(train_freq, 0.75), quantile_cont(train_freq, 0.90), quantile_cont(train_freq, 0.95),
  quantile_cont(train_freq, 0.99), MAX(train_freq), AVG(train_freq)
FROM user_freq_train
"""))

In [ ]:
# Cell 12 - Phân bố số tương tác/item và số lượng item có tương tác
item_pct = q("""
WITH f AS (
  SELECT split, parent_asin, COUNT(*) AS n_interactions
  FROM interactions_all
  GROUP BY split, parent_asin
), total AS (
  SELECT split, COUNT(*) AS n_items FROM f GROUP BY split
)
SELECT f.split, f.n_interactions, COUNT(*) AS items,
       ROUND(100.0 * COUNT(*) / MAX(total.n_items), 4) AS item_pct
FROM f JOIN total USING(split)
GROUP BY f.split, f.n_interactions
ORDER BY f.split, f.n_interactions
""")
display(item_pct[item_pct['n_interactions'] <= 30])

plt.figure(figsize=(14, 5))
plot_df = item_pct[item_pct['n_interactions'] <= 30]
sns.lineplot(data=plot_df, x='n_interactions', y='item_pct', hue='split', marker='o')
plt.title('% item theo số tương tác trong từng split, cắt tại 30')
plt.xlabel('Số tương tác/item')
plt.ylabel('% item')
plt.tight_layout()
plt.show()

display(q("""
SELECT
  'all_splits' AS scope,
  MIN(n_interactions) AS min_interactions,
  quantile_cont(n_interactions, 0.25) AS p25,
  quantile_cont(n_interactions, 0.50) AS p50,
  quantile_cont(n_interactions, 0.75) AS p75,
  quantile_cont(n_interactions, 0.90) AS p90,
  quantile_cont(n_interactions, 0.95) AS p95,
  quantile_cont(n_interactions, 0.99) AS p99,
  MAX(n_interactions) AS max_interactions,
  AVG(n_interactions) AS avg_interactions
FROM item_freq_all
UNION ALL
SELECT
  'train_only', MIN(train_freq), quantile_cont(train_freq, 0.25), quantile_cont(train_freq, 0.50),
  quantile_cont(train_freq, 0.75), quantile_cont(train_freq, 0.90), quantile_cont(train_freq, 0.95),
  quantile_cont(train_freq, 0.99), MAX(train_freq), AVG(train_freq)
FROM item_freq_train
"""))

In [ ]:
# Cell 13 - K-core candidate analysis cho user và item
thresholds = [1, 2, 3, 5, 10, 20, 50, 100]
rows = []
for scope, user_table, item_table, user_col, item_col in [
    ('all_splits', 'user_freq_all', 'item_freq_all', 'n_interactions', 'n_interactions'),
    ('train_only', 'user_freq_train', 'item_freq_train', 'train_freq', 'train_freq'),
]:
    for k in thresholds:
        users = q(f"SELECT COUNT(*) AS n FROM {user_table} WHERE {user_col} >= {k}").iloc[0, 0]
        items = q(f"SELECT COUNT(*) AS n FROM {item_table} WHERE {item_col} >= {k}").iloc[0, 0]
        inter_users = q(f"""
            SELECT COUNT(*) AS n
            FROM {'interactions_all' if scope == 'all_splits' else 'train'} i
            JOIN {user_table} u USING(reviewer_id)
            WHERE u.{user_col} >= {k}
        """).iloc[0, 0]
        inter_items = q(f"""
            SELECT COUNT(*) AS n
            FROM {'interactions_all' if scope == 'all_splits' else 'train'} i
            JOIN {item_table} it USING(parent_asin)
            WHERE it.{item_col} >= {k}
        """).iloc[0, 0]
        rows.append({'scope': scope, 'k': k, 'users_ge_k': users, 'items_ge_k': items, 'interactions_kept_by_user_k': inter_users, 'interactions_kept_by_item_k': inter_items})

kcore = pd.DataFrame(rows)
display(kcore)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=kcore, x='k', y='users_ge_k', hue='scope', marker='o', ax=axes[0])
axes[0].set_title('Số user có interactions >= k')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
sns.lineplot(data=kcore, x='k', y='items_ge_k', hue='scope', marker='o', ax=axes[1])
axes[1].set_title('Số item có interactions >= k')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 - Head/Mid/Tail theo fixed thresholds để so sánh ngưỡng
fixed_rules = [
    ('TAIL_1_2', 1, 2),
    ('TAIL_1_5', 1, 5),
    ('TAIL_1_10', 1, 10),
    ('MID_11_50', 11, 50),
    ('HEAD_51_PLUS', 51, None),
]
rows = []
for label, lo, hi in fixed_rules:
    cond = f'train_freq >= {lo}' if hi is None else f'train_freq BETWEEN {lo} AND {hi}'
    df = q(f"""
    SELECT
      '{label}' AS group_name,
      COUNT(*) AS items,
      SUM(train_freq) AS interactions,
      MIN(train_freq) AS min_freq,
      MAX(train_freq) AS max_freq,
      AVG(train_freq) AS avg_freq
    FROM item_freq_train
    WHERE {cond}
    """)
    rows.append(df)
fixed_hmt = pd.concat(rows, ignore_index=True)
total_items = q('SELECT COUNT(*) FROM item_freq_train').iloc[0, 0]
total_inter = q('SELECT COUNT(*) FROM train').iloc[0, 0]
fixed_hmt['item_pct'] = 100 * fixed_hmt['items'] / total_items
fixed_hmt['interaction_pct'] = 100 * fixed_hmt['interactions'] / total_inter
display(fixed_hmt)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=fixed_hmt, x='group_name', y='item_pct', ax=axes[0])
axes[0].set_title('Fixed threshold: % item')
axes[0].tick_params(axis='x', rotation=30)
sns.barplot(data=fixed_hmt, x='group_name', y='interaction_pct', ax=axes[1])
axes[1].set_title('Fixed threshold: % train interactions')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 - Head/Mid/Tail theo Pareto rank giống Silver: top 20% item = HEAD, next 10% = MID, còn lại = TAIL
freq = q("SELECT parent_asin, train_freq FROM item_freq_train ORDER BY train_freq DESC, parent_asin")
n_items = len(freq)
head_idx = int(n_items * 0.20)
mid_idx = int(n_items * 0.30)
head_cutoff = int(freq.iloc[min(head_idx, n_items - 1)]['train_freq']) if n_items else 0
tail_cutoff = int(freq.iloc[min(mid_idx, n_items - 1)]['train_freq']) if n_items else 0

def assign_pareto_group(x):
    if x >= head_cutoff:
        return 'HEAD'
    if x >= tail_cutoff:
        return 'MID'
    return 'TAIL'

freq['popularity_group'] = freq['train_freq'].map(assign_pareto_group)
pareto_hmt = freq.groupby('popularity_group', as_index=False).agg(
    items=('parent_asin', 'count'),
    interactions=('train_freq', 'sum'),
    min_freq=('train_freq', 'min'),
    max_freq=('train_freq', 'max'),
    avg_freq=('train_freq', 'mean'),
).sort_values('popularity_group')
pareto_hmt['item_pct'] = 100 * pareto_hmt['items'] / max(freq['parent_asin'].nunique(), 1)
pareto_hmt['interaction_pct'] = 100 * pareto_hmt['interactions'] / max(freq['train_freq'].sum(), 1)

display(Markdown(f'**Pareto cutoffs:** HEAD nếu `train_freq >= {head_cutoff}`, MID nếu `{tail_cutoff} <= train_freq < {head_cutoff}`, TAIL nếu `< {tail_cutoff}`.'))
display(pareto_hmt)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
order = ['HEAD', 'MID', 'TAIL']
sns.barplot(data=pareto_hmt, x='popularity_group', y='items', order=order, ax=axes[0])
axes[0].set_title('Pareto H/M/T: số item')
sns.barplot(data=pareto_hmt, x='popularity_group', y='interactions', order=order, ax=axes[1])
axes[1].set_title('Pareto H/M/T: số interactions train')
sns.boxplot(data=freq, x='popularity_group', y='train_freq', order=order, ax=axes[2])
axes[2].set_yscale('log')
axes[2].set_title('Train frequency theo group, log y')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 - Histogram log-scale và đường tích lũy để chọn ngưỡng Head/Mid/Tail
display(q("""
SELECT
  MIN(train_freq) min_freq,
  quantile_cont(train_freq, 0.50) p50,
  quantile_cont(train_freq, 0.70) p70,
  quantile_cont(train_freq, 0.80) p80,
  quantile_cont(train_freq, 0.90) p90,
  quantile_cont(train_freq, 0.95) p95,
  quantile_cont(train_freq, 0.99) p99,
  MAX(train_freq) max_freq
FROM item_freq_train
"""))

freq_sorted = freq.sort_values('train_freq', ascending=False).reset_index(drop=True)
freq_sorted['item_rank_pct'] = 100 * (np.arange(len(freq_sorted)) + 1) / len(freq_sorted)
freq_sorted['cum_interactions_pct'] = 100 * freq_sorted['train_freq'].cumsum() / freq_sorted['train_freq'].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(freq['train_freq'], bins=80, log_scale=(True, True), ax=axes[0])
axes[0].axvline(head_cutoff, color='red', linestyle='--', label=f'HEAD cutoff={head_cutoff}')
axes[0].axvline(tail_cutoff, color='orange', linestyle='--', label=f'MID/TAIL cutoff={tail_cutoff}')
axes[0].set_title('Histogram item train_freq, log-log')
axes[0].set_xlabel('train_freq')
axes[0].legend()

sns.lineplot(data=freq_sorted, x='item_rank_pct', y='cum_interactions_pct', ax=axes[1])
axes[1].axvline(20, color='red', linestyle='--', label='Top 20% items')
axes[1].axvline(30, color='orange', linestyle='--', label='Top 30% items')
axes[1].axhline(80, color='gray', linestyle=':', label='80% interactions')
axes[1].set_title('Cumulative interactions theo item rank')
axes[1].set_xlabel('% item theo rank giảm dần')
axes[1].set_ylabel('% interactions tích lũy')
axes[1].legend()
plt.tight_layout()
plt.show()

display(freq_sorted.head(20))

In [ ]:
# Cell 17 - Val/Test cold-start item so với train và group H/M/T của item warm
con.register('item_popularity_df', freq[['parent_asin', 'train_freq', 'popularity_group']])
con.execute("CREATE OR REPLACE VIEW item_popularity AS SELECT * FROM item_popularity_df")

cold = q("""
WITH labeled AS (
  SELECT i.split, i.parent_asin,
         COALESCE(p.train_freq, 0) AS train_freq,
         COALESCE(p.popularity_group, 'COLD_START') AS popularity_group
  FROM interactions_all i
  LEFT JOIN item_popularity p USING(parent_asin)
  WHERE i.split IN ('val', 'test')
)
SELECT split, popularity_group, COUNT(*) AS interactions, COUNT(DISTINCT parent_asin) AS items
FROM labeled
GROUP BY split, popularity_group
ORDER BY split, popularity_group
""")
cold['interaction_pct_in_split'] = cold.groupby('split')['interactions'].transform(lambda s: 100 * s / s.sum())
display(cold)

plt.figure(figsize=(10, 5))
sns.barplot(data=cold, x='split', y='interaction_pct_in_split', hue='popularity_group')
plt.title('Val/Test: % interactions theo popularity group dựa trên train')
plt.ylabel('% interactions trong split')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 18 - Metadata EDA: coverage, numeric fields, category/store
meta_overview = q("""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT parent_asin) AS distinct_items,
  COUNT(*) - COUNT(DISTINCT parent_asin) AS duplicate_parent_asin_rows,
  SUM(CASE WHEN title IS NULL OR trim(title) = '' THEN 1 ELSE 0 END) AS empty_title,
  SUM(CASE WHEN main_category IS NULL OR trim(main_category) = '' THEN 1 ELSE 0 END) AS empty_main_category,
  SUM(CASE WHEN store IS NULL OR trim(store) = '' THEN 1 ELSE 0 END) AS empty_store,
  SUM(CASE WHEN price IS NULL OR price <= 0 THEN 1 ELSE 0 END) AS non_positive_price,
  AVG(price) AS avg_price,
  AVG(average_rating) AS avg_average_rating,
  AVG(rating_number) AS avg_rating_number
FROM meta
""")
display(meta_overview)

coverage = q("""
WITH all_items AS (SELECT DISTINCT parent_asin FROM interactions_all),
train_items AS (SELECT DISTINCT parent_asin FROM train),
meta_items AS (SELECT DISTINCT parent_asin FROM meta)
SELECT 'all_interaction_items' AS scope, COUNT(*) AS items, SUM(CASE WHEN m.parent_asin IS NOT NULL THEN 1 ELSE 0 END) AS items_with_meta
FROM all_items a LEFT JOIN meta_items m USING(parent_asin)
UNION ALL
SELECT 'train_items', COUNT(*), SUM(CASE WHEN m.parent_asin IS NOT NULL THEN 1 ELSE 0 END)
FROM train_items a LEFT JOIN meta_items m USING(parent_asin)
""")
coverage['meta_coverage_pct'] = 100 * coverage['items_with_meta'] / coverage['items']
display(coverage)

display(Markdown('### Top main_category'))
display(q("""
SELECT main_category, COUNT(*) AS items
FROM meta
GROUP BY main_category
ORDER BY items DESC
LIMIT 20
"""))

display(Markdown('### Top store'))
display(q("""
SELECT store, COUNT(*) AS items
FROM meta
GROUP BY store
ORDER BY items DESC
LIMIT 20
"""))

In [ ]:
# Cell 19 - Review text/title noise: độ dài text, empty text, sample text dài/ngắn
text_stats = q("""
SELECT split,
       AVG(length(coalesce(review_title, ''))) AS avg_title_len,
       AVG(length(coalesce(review_text, ''))) AS avg_text_len,
       quantile_cont(length(coalesce(review_text, '')), 0.5) AS text_len_p50,
       quantile_cont(length(coalesce(review_text, '')), 0.9) AS text_len_p90,
       quantile_cont(length(coalesce(review_text, '')), 0.99) AS text_len_p99,
       SUM(CASE WHEN length(trim(coalesce(review_text, ''))) = 0 THEN 1 ELSE 0 END) AS empty_text_rows
FROM interactions_all
GROUP BY split
ORDER BY split
""")
display(text_stats)

sample_len = q("""
SELECT split, length(coalesce(review_text, '')) AS review_text_len
FROM interactions_all
""")
plt.figure(figsize=(14, 5))
sns.histplot(data=sample_len, x='review_text_len', hue='split', bins=100, log_scale=(True, True))
plt.title('Phân bố độ dài review_text, log-log')
plt.xlabel('Số ký tự review_text')
plt.tight_layout()
plt.show()

display(Markdown('### Sample review_text dài nhất'))
display(q("""
SELECT split, reviewer_id, parent_asin, rating, length(review_text) AS text_len, review_title, review_text
FROM interactions_all
ORDER BY length(review_text) DESC NULLS LAST
LIMIT 10
"""))

In [ ]:
# Cell 20 - Top users/items và item metadata tương ứng
display(Markdown('### Top users theo số tương tác all splits'))
display(q("""
SELECT reviewer_id, COUNT(*) AS interactions,
       SUM(CASE WHEN split='train' THEN 1 ELSE 0 END) AS train_interactions,
       SUM(CASE WHEN split='val' THEN 1 ELSE 0 END) AS val_interactions,
       SUM(CASE WHEN split='test' THEN 1 ELSE 0 END) AS test_interactions
FROM interactions_all
GROUP BY reviewer_id
ORDER BY interactions DESC
LIMIT 20
"""))

display(Markdown('### Top train items kèm metadata'))
display(q("""
SELECT f.parent_asin, f.train_freq, p.popularity_group, m.title, m.main_category, m.store, m.average_rating, m.rating_number
FROM item_freq_train f
LEFT JOIN item_popularity p USING(parent_asin)
LEFT JOIN meta m USING(parent_asin)
ORDER BY f.train_freq DESC
LIMIT 30
"""))

In [ ]:
# Cell 21 - Bảng khuyến nghị ngưỡng popularity dựa trên nhiều tiêu chí
candidate_rows = []
for tail_max in [2, 3, 5, 10, 20]:
    for head_min in [20, 50, 100]:
        if tail_max >= head_min:
            continue
        tmp = freq.copy()
        tmp['group'] = np.where(tmp['train_freq'] >= head_min, 'HEAD', np.where(tmp['train_freq'] <= tail_max, 'TAIL', 'MID'))
        g = tmp.groupby('group').agg(items=('parent_asin', 'count'), interactions=('train_freq', 'sum')).reset_index()
        row = {'rule': f'TAIL<= {tail_max}, HEAD>= {head_min}'}
        for _, r in g.iterrows():
            row[f"{r['group']}_items_pct"] = 100 * r['items'] / len(tmp)
            row[f"{r['group']}_interactions_pct"] = 100 * r['interactions'] / tmp['train_freq'].sum()
        candidate_rows.append(row)

candidates = pd.DataFrame(candidate_rows).fillna(0)
display(candidates)

display(Markdown(f"""
### Gợi ý đọc kết quả

- Nếu muốn bám pipeline Silver hiện tại: dùng Pareto cutoff ở Cell 15: `HEAD >= {head_cutoff}`, `MID >= {tail_cutoff} and < {head_cutoff}`, `TAIL < {tail_cutoff}`.
- Nếu muốn ngưỡng dễ diễn giải cho báo cáo: so sánh bảng fixed threshold ở Cell 14 và bảng candidate ở cell này.
- Ngưỡng tốt nên tránh để `HEAD` chiếm quá nhiều item nhưng quá ít tương tác, hoặc `TAIL` chiếm gần toàn bộ tương tác. Hãy ưu tiên cân bằng theo mục tiêu model: warm long-tail thì `TAIL` nên còn đủ interactions để train/evaluate.
"""))